##FORMAT DATASET

In [0]:
from pyspark.sql.functions import col, regexp_replace

###Formatting olist_order_reviews_dataset

In [0]:
# ====================================================================
# Read CSV File From Volume
# ====================================================================

input_path = "/Volumes/olist_lakehouse/landing/olist_row_data/olist_order_reviews_dataset.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(input_path)
)

# ====================================================================
# Replace ',' with ';' in Review Columns
# ====================================================================

df_formatted = (
    df
    .withColumn(
        "review_comment_message",
        regexp_replace(
            col("review_comment_message"),
            ",",
            ";"
        )
    )
    .withColumn(
        "review_comment_title",
        regexp_replace(
            col("review_comment_title"),
            ",",
            ";"
        )
    )
)

# ====================================================================
# Write Formatted CSV File To Volume
# ====================================================================

output_path = "/Volumes/olist_lakehouse/formatted/formatted_row_data/olist_order_reviews_dataset"

(
    df_formatted
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(output_path)
)

# ====================================================================
# Validation
# ====================================================================

display(df_formatted.limit(10))

print("Formatted CSV file written successfully")
print(f"Output Path: {output_path}")

###Move other files to formatted [volume](url)

In [0]:
# Databricks Notebook
# ====================================================================
# COPY RAW CSV FILES FROM LANDING TO FORMATTED LAYER
# ====================================================================

# ====================================================================
# Source and Target Volumes
# ====================================================================

source_base_path = "/Volumes/olist_lakehouse/landing/olist_row_data/"
target_base_path = "/Volumes/olist_lakehouse/formatted/formatted_row_data/"

# ====================================================================
# File List
# ====================================================================

file_list = [

    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"

]

# ====================================================================
# Read and Write Each File
# ====================================================================

for file_name in file_list:

    print(f"Processing File: {file_name}")

    input_path = source_base_path + file_name

    output_path = (
        target_base_path
        + file_name.replace(".csv", "")
    )

    # Read CSV File
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(input_path)
    )

    # Write CSV File
    (
        df.write
        .mode("overwrite")
        .option("header", True)
        .csv(output_path)
    )

    print(f"SUCCESS: {file_name} copied successfully")
    print(f"Output Path: {output_path}")
    print("--------------------------------------------------")

print("All files processed successfully")